# Task 6 · round 4: leaves + prediction scale (plan_t6.md)

**Round 3 (`task6_tune.ipynb`):** with 3-seed bags, noise is ~0.04% (seeds 0–2 vs 3–5);
only `max_leaf_nodes=127` helped (−0.30% vs the shipped settings, same seeds); the best
constant scale on A was **×1.05 (−0.31%)**.

**Why a scale:** the model learns from the gap one step *before* the one it predicts, and
held-out gaps grow step by step (median 43.9 days on backtest B → 49.1 on A; the test is one
step later again, with t_last median 2016-09 vs 2016-04 on A). So it predicts too short.
Backtest A has exactly the test's relation (fit on the gap before, predict the next), so its
best scale is the estimate; B checks that it is stable.

**Ship if** leaves + scale beat the shipped settings on **both** backtests.

In [1]:
%load_ext autoreload
%autoreload 2

import time

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

from assignment_1.modules.models import GapModel, GapRegressor, split_last
from assignment_1.modules.tasks.task6 import TASK6_GAP_PARAMS, TASK6_REGRESSOR_PARAMS
from assignment_1.modules.util import load_csv

DATA_DIR = "../data"  # notebook runs from notebooks/
train = load_csv("task_6_train.csv", DATA_DIR)
trA, valA = split_last(train)
trB, valB = split_last(trA)

## S1. Cache features for both backtests (slow: ~1 min each)

In [2]:
def cache(tr: pd.DataFrame, val: pd.DataFrame) -> dict:
    m = GapRegressor(base=GapModel(**TASK6_GAP_PARAMS), **TASK6_REGRESSOR_PARAMS)
    hist_b, last_b = split_last(tr)
    return {"X": m.features(hist_b, last_b), "y": np.log1p(last_b["target"].to_numpy()),
            "Xv": m.features(tr, val), "yv": val["target"].to_numpy()}

t0 = time.time()
C = {"A": cache(trA, valA), "B": cache(trB, valB)}
print(f"cached in {time.time() - t0:.0f}s")

cached in 116s


## S2. Bagged predictions per leaf count (3 seeds, as in round 3)

In [3]:
BASE = dict(loss="absolute_error", learning_rate=0.03, max_iter=500, min_samples_leaf=40)
LEAVES = (31, 63, 127, 255)   # 31 = shipped

def bag_pred(b: str, leaves: int, seeds=(0, 1, 2)) -> np.ndarray:
    c = C[b]
    return np.expm1(np.mean([
        HistGradientBoostingRegressor(**BASE, max_leaf_nodes=leaves, random_state=s)
        .fit(c["X"], c["y"]).predict(c["Xv"]) for s in seeds], axis=0))

P = {(b, n): bag_pred(b, n) for b in "AB" for n in LEAVES}

## S3. MAE by leaf count and scale
Rows: leaf count; columns: scale. Δ% is against the shipped settings (31 leaves, ×1.00) on the
same backtest.

In [4]:
SCALES = np.round(np.arange(1.00, 1.161, 0.02), 2)

def mae_table(b: str) -> pd.DataFrame:
    y = C[b]["yv"]
    T = pd.DataFrame({s: [np.mean(np.abs(s * P[(b, n)] - y)) for n in LEAVES] for s in SCALES},
                     index=pd.Index(LEAVES, name="leaves"))
    return T

for b in "AB":
    T = mae_table(b)
    ref = T.loc[31, 1.0]
    print(f"backtest {b}: shipped settings {ref:,.0f}")
    display((100 * (T / ref - 1)).round(2))

backtest A: shipped settings 7,116,635


,1.00,1.02,1.04,1.06,1.08,1.10,1.12,1.14,1.16
leaves,,,,,,,,,
31,0.00,-0.18,-0.29,-0.32,-0.27,-0.17,-0.01,0.20,0.47
63,-0.17,-0.40,-0.55,-0.62,-0.62,-0.55,-0.43,-0.25,-0.01
127,-0.30,-0.53,-0.68,-0.75,-0.75,-0.68,-0.55,-0.36,-0.11
255,-0.10,-0.35,-0.54,-0.64,-0.68,-0.65,-0.55,-0.40,-0.20


backtest B: shipped settings 7,097,583


,1.00,1.02,1.04,1.06,1.08,1.10,1.12,1.14,1.16
leaves,,,,,,,,,
31,0.00,-0.09,-0.11,-0.06,0.06,0.23,0.46,0.74,1.08
63,-0.24,-0.35,-0.39,-0.37,-0.28,-0.14,0.05,0.30,0.60
127,-0.26,-0.40,-0.47,-0.47,-0.42,-0.30,-0.13,0.09,0.36
255,-0.25,-0.39,-0.46,-0.46,-0.41,-0.29,-0.12,0.10,0.37


## S4. Best scale per leaf count, on each backtest
Fine grid. If A and B agree, the scale is stable and A's value is the one to ship.

In [5]:
fine = np.round(np.arange(0.98, 1.20, 0.005), 3)
rows = []
for n in LEAVES:
    row = {"leaves": n}
    for b in "AB":
        y = C[b]["yv"]
        m = np.array([np.mean(np.abs(s * P[(b, n)] - y)) for s in fine])
        ref = np.mean(np.abs(P[(b, 31)] - y))
        row[f"best_scale_{b}"] = fine[m.argmin()]
        row[f"Δ%_{b} at own best"] = 100 * (m.min() / ref - 1)
    rows.append(row)
pd.DataFrame(rows).round(3)

,leaves,best_scale_A,Δ%_A at own best,best_scale_B,Δ%_B at own best
0,31,1.06,-0.316,1.035,-0.107
1,63,1.07,-0.629,1.045,-0.391
2,127,1.07,-0.759,1.050,-0.477
3,255,1.08,-0.684,1.050,-0.466


## S5. Chosen setting on both backtests
Set `LEAVES_BEST` and `SCALE` (A's best scale for that leaf count, rounded) from S4.

In [6]:
LEAVES_BEST, SCALE = 127, 1.05   # <- from S4
out = {}
for b in "AB":
    y = C[b]["yv"]
    ref = np.mean(np.abs(P[(b, 31)] - y))
    new = np.mean(np.abs(SCALE * P[(b, LEAVES_BEST)] - y))
    out[b] = {"shipped": ref, "new": new, "Δ%": 100 * (new / ref - 1)}
pd.DataFrame(out).T.round(2)

,shipped,new,Δ%
A,7116635.43,7064959.89,-0.73
B,7097583.23,7063749.77,-0.48


## Decision
If S5 wins on both: in `tasks/task6.py` set
```python
TASK6_REGRESSOR_PARAMS = {
    "coread": True, "coread_k": 5, "n_bag": 5, "scale": SCALE,
    "regressor": HistGradientBoostingRegressor(
        loss="absolute_error", learning_rate=0.03, max_iter=500, min_samples_leaf=40,
        max_leaf_nodes=LEAVES_BEST, random_state=0),
}
```
then `make check TASKS=6`, `make changes TASKS=6`, upload. The test is one step further along
than A, so the ideal test scale may be a little above A's; a second upload with a slightly
larger scale is one way to check, at the cost of tuning on the public board.